# P2.5 · Multilabel classifier (per-class BCE, micro vs macro F1, per-class thresholds)

**ID:** P2.5 · **Target:** 20 min · **Plan week:** 2 · **Part:** P (PyTorch first principles)

**The drill.** Each example can carry any subset of C labels. Train an MLP with one logit per class and
`BCEWithLogitsLoss` on a float 0/1 target matrix, compute micro and macro F1 **by hand** (checked against
scikit-learn), and tune one threshold per class on val.

**Must include (from the plan):** per-class BCE, micro vs macro F1. Also (guide §P1 traps): targets are a
**float** 0/1 matrix of shape (N, C), and thresholds are chosen **per class at eval**. Initial-loss check,
overfit-one-batch, whole-set eval under `eval()` + `no_grad()`, test touched once.

**Rules.** No docs, no LLM. Each solution cell keeps only its spec. The self-check names the variables it expects.

## Sources
- scikit-learn `make_multilabel_classification` (generative process: number of labels ~ Poisson, features are
  word counts): https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_multilabel_classification.html
- scikit-learn *Multiclass and multioutput algorithms* / `f1_score(average="micro" | "macro" | "samples")`
  (reference for the checks; `zero_division`): https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
- G. Tsoumakas, I. Katakis, I. Vlahavas, *Mining Multi-label Data*, in Data Mining and Knowledge Discovery Handbook,
  Springer 2010, pp. 667-685, DOI 10.1007/978-0-387-09823-4_34 — https://doi.org/10.1007/978-0-387-09823-4_34
  (example-based vs label-based metrics; micro vs macro averaging).
- Z. C. Lipton, C. Elkan, B. Narayanaswamy, *Thresholding Classifiers to Maximize F1 Score*, ECML-PKDD 2014,
  arXiv:1402.1892 — https://arxiv.org/abs/1402.1892 (micro/macro F1 thresholding in the multilabel setting).
- `torch.nn.BCEWithLogitsLoss` docs (element-wise BCE, `pos_weight` of shape (C,) for multilabel):
  https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from prep import Session, check, check_close, seed_everything

s = Session("P2.5", target_min=20)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. make_multilabel_classification(n_samples=8000, n_features=40, n_classes=10, n_labels=2,
#    length=150, allow_unlabeled=True, random_state=0) -> X (counts), Y (N, 10) 0/1.
#    Inspect: label frequency per class (prevalence), mean labels per example, fraction with no label.
#    Split 70/15/15 (random_state=0; plain split, stratifying multilabel data needs iterative stratification).
#    Features: log1p, then standardize with train stats. Tensors: X_* float32; Y_* FLOAT32 (N, 10). Keep Y_*_np.
# YOUR CODE HERE


In [ ]:
# 2. F1 BY HAND. Inputs: 0/1 numpy arrays (N, C).
#    per_class_counts(Y, P) -> tp, fp, fn, each shape (C,).
#    f1_micro(Y, P): pool tp/fp/fn over ALL classes, then F1 = 2TP / (2TP + FP + FN).
#    f1_macro(Y, P): F1 per class (0 when 2TP+FP+FN == 0, sklearn's zero_division=0), then the unweighted mean.
#    f1_samples(Y, P): F1 per EXAMPLE (row), then the mean (same zero-division rule).
# YOUR CODE HERE


In [ ]:
# 3. MODEL + LOSS. `MLP`: 40 -> 128 -> 128 -> C, ReLU, Dropout(0.1); returns (B, C) logits. make_model() on `dev`.
#    loss_fn = nn.BCEWithLogitsLoss() (mean over all N*C entries). DataLoaders: train batch 128 shuffle, val/test 2048.
#    Sanity: `init_loss` on val (expect ≈ ln 2 for every entry); overfit one batch of 64 (fresh model, eval mode,
#    Adam lr=1e-2, 300 steps) -> `overfit_loss` < 0.05.
# YOUR CODE HERE


In [ ]:
# 4. TRAIN + EVAL. AdamW lr=1e-3, wd=1e-2, 30 epochs. `probs(model, dl) -> numpy (N, C)` sigmoid probabilities
#    (eval, no_grad, whole set). Each epoch record train loss and val micro-F1 at threshold 0.5 in `history`
#    ("train_loss", "val_micro"). Keep the best-val-micro state and load it at the end.
# YOUR CODE HERE


In [ ]:
# 5. PER-CLASS THRESHOLDS. On val, for each class c choose t_c from the grid np.linspace(0.05, 0.95, 19) (it must
#    include 0.5) maximizing that class's F1 -> `thr` (C,). Report on val AND test (test once):
#    micro / macro / samples F1 at 0.5 and at `thr` -> dicts `val_05`, `val_thr`, `test_05`, `test_thr`
#    with keys "micro", "macro", "samples". Also print the per-class F1 + prevalence table for test.
# YOUR CODE HERE


## Self-check
Expects: `Y_train`, `f1_micro`, `f1_macro`, `f1_samples`, `model`, `loss_fn`, `init_loss`, `overfit_loss`,
`thr`, `val_05`, `val_thr`, `test_thr`.

In [ ]:
from sklearn.metrics import f1_score

rng = np.random.default_rng(0)
prev = np.array([0.5, 0.3, 0.1, 0.03, 0.0])          # imbalanced classes, one with NO positives
Yr = (rng.random((400, 5)) < prev).astype(int)
Pr = np.where(rng.random((400, 5)) < 0.8, Yr, 1 - Yr)  # 80 %-accurate predictions
Pr[:, 4] = 0                                           # nothing predicted for the empty class either
for avg, fn in [("micro", f1_micro), ("macro", f1_macro), ("samples", f1_samples)]:
    check_close(f"f1 {avg} == sklearn", fn(Yr, Pr), f1_score(Yr, Pr, average=avg, zero_division=0))
check("micro != macro on imbalanced labels", abs(f1_micro(Yr, Pr) - f1_macro(Yr, Pr)) > 0.05)
check("targets are a FLOAT 0/1 matrix (N, C)", Y_train.dtype == torch.float32 and Y_train.shape[1] == C)
check("loss is BCEWithLogits", isinstance(loss_fn, nn.BCEWithLogitsLoss))
check("model returns (B, C) logits", model(X_train[:4].to(dev)).shape == (4, C))
check("no Sigmoid in the model", not any(isinstance(m, nn.Sigmoid) for m in model.modules()))
check("initial loss ~ ln 2", abs(init_loss - math.log(2)) < 0.1, f"got {init_loss:.3f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
check("one threshold per class", np.shape(thr) == (C,))
check("per-class thresholds can't hurt val macro-F1", val_thr["macro"] >= val_05["macro"] - 1e-9, f"{val_thr} vs {val_05}")
check("test micro-F1 > 0.70", test_thr["micro"] > 0.70, f"got {test_thr['micro']:.3f}")
check("test macro-F1 > 0.60", test_thr["macro"] > 0.60, f"got {test_thr['macro']:.3f}")

## Say it out loud (P3)
1. Why BCE per class and not softmax + CE? When would softmax be right even with "many" labels?
2. Why must the targets be float, and what shape is the loss before reduction?
3. Micro vs macro F1: which one does a frequent class dominate? Which would you report for a long-tailed label set, and why both?
4. Why tune thresholds per class, and why on val? What is the risk for a class with 20 val positives?
5. What is "samples" F1, and how is it different from subset accuracy (exact match) and Hamming loss?
6. Why is the initial loss ln 2 even though most labels are 0? What init would give a lower initial loss?
7. How would you handle label imbalance in the loss (shape of `pos_weight`)?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up